In [1]:
import torch
import matplotlib.pyplot as plt

from transformers import AutoVideoProcessor, AutoModel, AutoImageProcessor
from transformers import pipeline

import os
import torch
import numpy as np

from torchcodec.decoders import VideoDecoder
from IPython.display import Video
import pandas as pd

import torchvision
import gc


In [2]:
device = "cpu"

In [3]:
from src.clusters import HierarchicalCluster

In [4]:
dataset_path = '/data/scratch/hmz574/scratch-aid/videos/training_test_videos/'
output_path = '/data/scratch/hmz574/scratch-aid/videos/training_test_videos_embeds/'
file_list = os.listdir(dataset_path)

In [5]:
train_list = ['V'+ str(x + 1) + '.mp4' for x in range (32)]


In [6]:
frame_rate = 30.0
offset = int(frame_rate * 0.1) # seconds
duration = int(frame_rate * 2) #seconds
video_name = 'V20'

In [7]:
dataset_path + video_name + '.mp4'


'/data/scratch/hmz574/scratch-aid/videos/training_test_videos/V20.mp4'

In [8]:
decoder = VideoDecoder(dataset_path + video_name + '.mp4', device='cpu')
decoder[0:10].shape

torch.Size([10, 3, 720, 720])

In [9]:
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small', use_fast=True)
model = AutoModel.from_pretrained('facebook/dinov2-small')


In [10]:
inputs = processor(images = decoder[0:10], return_tensors='pt')
inputs['pixel_values'].shape

torch.Size([10, 3, 224, 224])

In [11]:
from torchvision.io import read_video
def video_loader(video_path, start, end):
    frames, _, _ = read_video(str(video_path), output_format="TCHW",start_pts=start,end_pts=end, pts_unit='sec')
    return frames


In [12]:
v = video_loader(dataset_path + video_name + '.mp4',start=1199, end=1201)
v.shape

/data/home/hmz574/.local/lib/python3.12/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


torch.Size([33, 3, 720, 720])

In [13]:
v.shape

torch.Size([33, 3, 720, 720])

In [14]:
inputs = processor(images = v, return_tensors='pt')
outputs = model(**inputs)
outputs['last_hidden_state'][:,0].shape

torch.Size([33, 384])

In [15]:
batch_size = 30
total_frames = len(decoder)

batches = total_frames//batch_size
print(f"Frames: {total_frames}, Batches: {batches}")
embeddings = []

Frames: 36001, Batches: 1200


In [ ]:
for i in range(batches):
    gc.collect()
    if i % 10 == 0:
        print(f"batch: {i}")
    frame_start = i * batch_size
    inputs = processor(images = decoder[frame_start:frame_start+batch_size], return_tensors='pt')
    print(f"{frame_start}")
    outputs = model(**inputs)
    last_hidden_state = outputs['last_hidden_state']
    cls_token = last_hidden_state[:,0]
    #torch.save(cls_token, output_path + video_name + '.mp4.' + str(i) + '.pt')
    gc.collect()
    #embeddings.append(cls_token)

batch: 0
0
30
60
90
120
150
180
210
240


In [ ]:
embeddings[8].shape

In [ ]:
i = 0
while True:
    gc.collect()
    v = video_loader(dataset_path + video_name + '.mp4',start=i, end=i+1)
    if v.shape[0] == 1:
        break
    inputs = processor(images = v, return_tensors='pt')
    outputs = model(**inputs)
    last_hidden_state = outputs['last_hidden_state']
    cls_token = last_hidden_state[:,0]
    #embeddings.append(cls_token)
    #torch.save(cls_token, output_path + video_name + '.mp4.' + str(i) + '.pt')
    gc.collect()
    i+=1
    if i % 2 == 0:
        print(i)

2
4
6
8
10
12
14
16
18
20
22


/bin/bash: line 1: nvstats: command not found
